## Technical Coding Challenge (Python Notebook)

### Create a Jupyter Notebook named week3_multi_source_pipeline.ipynb.
#### 1. Source 1 (Internal): Load your cleaned operational dataset from Week 2 (or use the provided mock_ops.csv ).

In [506]:
import pandas as pd


mock_df = pd.read_csv("mock_ops.csv", index_col=0)

#### 2. Source 2 (External API): Write a script using requests to fetch real-time data relevant to your operations (e.g., Weather for logistics, Currency Rates for fintech, Commodity Prices for retail). Handle errors (try/except) if the API fails.

In [507]:
import openmeteo_requests

# import pandas as pd
import requests_cache
from retry_requests import retry


cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)


url = "https://api.open-meteo.com/v1/forecast"

zone_coordinates = {
	"Zone_North": {
	"name": "Turkana",
	"coordinates": {
		"latitude": 0.6649,
		"longitude": 37.8913
	}},
	"Zone_South": {
	"name": "Mombasa",
	"coordinates": {
		"latitude": -4.0547,
		"longitude": 39.6636
	}},
	"Zone_Central": {
	"name": "Nairobi",
	"coordinates": {
		"latitude": -1.2833,
		"longitude": 36.8167
	}},
	"Zone_East": {
	"name": "Makueni",
	"coordinates": {
		"latitude": -2.1026,
		"longitude": 38.1301
	}},
	"Zone_West": {
	"name": "Kisumu",
	"coordinates": {
		"latitude": -0.1022,
		"longitude": 34.7617
	}},
}

hourly_dataframes = {}

for key in zone_coordinates:
	zones = zone_coordinates[key]
	print(f"zone name: {key}")
	print(f"location name: {zones["name"]}")
	locations = zones["coordinates"]
	print(locations)
	for coordinate in locations:
		latitude = locations["latitude"]
		longitude = locations["longitude"]

		params = {
			"latitude": latitude,
			"longitude": longitude,
			"hourly": ["temperature_2m", "relative_humidity_2m", "rain", "weather_code", "precipitation", "apparent_temperature", "surface_pressure"],
			"timezone": "Africa/Nairobi",
			"past_days": 31,
		}
	
	# if any single statement throws an error, the program immediately skips the remainder of the try block and jumps to the first matching except statement
	try:
		responses = openmeteo.weather_api(url, params = params)

		response = responses[0]
		print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
		print(f"Elevation: {response.Elevation()} m asl")
		print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
		print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")


		hourly = response.Hourly()
		hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
		hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
		hourly_rain = hourly.Variables(2).ValuesAsNumpy()
		hourly_weather_code = hourly.Variables(3).ValuesAsNumpy()
		hourly_precipitation = hourly.Variables(4).ValuesAsNumpy()
		hourly_apparent_temperature = hourly.Variables(5).ValuesAsNumpy()
		hourly_surface_pressure = hourly.Variables(6).ValuesAsNumpy()

		hourly_data = {
			"date": pd.date_range(
				start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
				end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
				freq = pd.Timedelta(seconds = hourly.Interval()),
				inclusive = "left"
			).tz_convert(response.Timezone().decode()).tz_localize(None)
		}
		# print("Start Time:", hourly.Time())
		# print("End Time:", hourly.TimeEnd())
		# print("Interval:", hourly.Interval())

		hourly_data["temperature_2m"] = hourly_temperature_2m
		hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
		hourly_data["rain"] = hourly_rain
		hourly_data["weather_code"] = hourly_weather_code
		hourly_data["precipitation"] = hourly_precipitation
		hourly_data["apparent_temperature"] = hourly_apparent_temperature
		hourly_data["surface_pressure"] = hourly_surface_pressure

		hourly_dataframe = pd.DataFrame(data = hourly_data)
		hourly_dataframe['date'] = pd.to_datetime(hourly_dataframe['date'])
		hourly_dataframe = hourly_dataframe.set_index('date')
		hourly_dataframe = hourly_dataframe.loc[pd.date_range('2026-06-25 00:00:00', '2026-07-01 22:00:00', freq='1h')]
		hourly_dataframe.index.name = "timestamp"
		hourly_dataframes[key] = hourly_dataframe
		print(f"\n{key}_Hourly data\n", hourly_dataframe)

		hourly_dataframe.to_csv(f"{key}_hourly_weather.csv")

		# for key, df in hourly_dataframes.items():
		#     print(f"\nKey: {key}")
		#     print(df)

	except Exception as e:
		print(f"Unexpected error: {e}")



zone name: Zone_North
location name: Turkana
{'latitude': 0.6649, 'longitude': 37.8913}
Coordinates: 0.6678383350372314°N 37.9058837890625°E
Elevation: 821.0 m asl
Timezone: b'Africa/Nairobi'b'GMT+3'
Timezone difference to GMT+0: 10800s

Zone_North_Hourly data
                      temperature_2m  relative_humidity_2m  rain  weather_code  \
timestamp                                                                       
2026-06-25 00:00:00       22.900000             57.251568   0.0           3.0   
2026-06-25 01:00:00       22.049999             63.281109   0.0           3.0   
2026-06-25 02:00:00       21.150000             68.605835   0.0           3.0   
2026-06-25 03:00:00       21.049999             65.347382   0.0           2.0   
2026-06-25 04:00:00       20.750000             67.648094   0.0           1.0   
...                             ...                   ...   ...           ...   
2026-07-01 18:00:00       27.750000             38.778297   0.0           0.0   
2026-07-0

#### 3. Source 3 (Database): Create a small SQLite database (using sqlalchemy ) containing supplementary data (e.g., Holiday_Calendar or Employee_Shifts ). Write a SQL query using JOIN and GROUP BY to extract summary info, loading it directly into a DataFrame via pd.read_sql .

In [508]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///openmeteo.db")

for key, df in hourly_dataframes.items():
    df.to_sql(
        name=key,
        con=engine,
        if_exists="replace",
        # index=False
    )


# mock_df = pd.read_csv("mock_ops.csv", parse_dates=True, index_col=0)  # use the very first column of the CSV file as the index after parsing
# mock_df = pd.read_csv("mock_ops.csv", index_col=0)
mock_df.index = pd.to_datetime(mock_df.index)
hourly_avg_df = (mock_df.groupby("Zone").resample('1h').mean(numeric_only=True).reset_index())
print(f"\nHOURLY RESAMPLED & AVERAGED\n{hourly_avg_df.head()}")
hourly_avg_df.to_csv("hourly_ops.csv", index=False)
hourly_avg_df.to_sql(name="Ops_Sensor", con=engine, if_exists="replace", index=False)





HOURLY RESAMPLED & AVERAGED
           Zone           timestamp  Pressure_PSI  Temperature_C  \
0  Zone_Central 2026-06-25 00:00:00    173.492287      70.962801   
1  Zone_Central 2026-06-25 01:00:00    222.558183      70.907593   
2  Zone_Central 2026-06-25 02:00:00    176.288000      73.662030   
3  Zone_Central 2026-06-25 03:00:00    212.373500      61.534961   
4  Zone_Central 2026-06-25 04:00:00    185.760994      68.488352   

   Flow_Rate_LPM  
0     913.562985  
1    1231.772427  
2     966.813758  
3    1084.301103  
4    1037.571987  


835

In [509]:
from sqlalchemy import text

tables = ['Zone_Central', 'Zone_East', 'Zone_North', 'Zone_South', 'Zone_West']

with engine.begin() as conn:
    for tb in tables:
        result = conn.execute(text(f"PRAGMA table_info({tb})"))
        
        columns = [row[1] for row in result]
        
        if "Zone" not in columns:
            conn.execute(text(f"ALTER TABLE {tb} ADD COLUMN Zone TEXT"))
        conn.execute(text(f"UPDATE {tb} SET Zone = '{tb}'"))

    conn.execute(text("DROP TABLE IF EXISTS Zones_Weather"))
    

    union_query = "\nUNION ALL\n".join(
        [f"SELECT * FROM {tb}" for tb in tables]
    )

    conn.execute(text(f"CREATE TABLE Zones_Weather AS {union_query}"))
    conn.commit()

expected = (
pd.read_sql("SELECT COUNT(*) FROM Zone_North", engine).iloc[0, 0]
+ pd.read_sql("SELECT COUNT(*) FROM Zone_South", engine).iloc[0, 0]
+ pd.read_sql("SELECT COUNT(*) FROM Zone_Central", engine).iloc[0, 0]
+ pd.read_sql("SELECT COUNT(*) FROM Zone_East", engine).iloc[0, 0]
+ pd.read_sql("SELECT COUNT(*) FROM Zone_West", engine).iloc[0, 0]
)

actual = pd.read_sql("SELECT COUNT(*) FROM Zones_Weather", engine).iloc[0, 0]

print(expected)
print(actual)
print(expected == actual)

835
835
True


In [510]:
query = """
SELECT
    Zone,
    AVG(temperature_2m) AS avg_tmp,
    AVG(relative_humidity_2m) AS avg_humd
FROM Zones_Weather
GROUP BY Zone
"""

summary = pd.read_sql(query, engine)

print(summary)

           Zone    avg_tmp   avg_humd
0  Zone_Central  18.054192  71.433564
1     Zone_East  23.941617  56.541092
2    Zone_North  25.413174  51.458719
3    Zone_South  25.609581  73.542758
4     Zone_West  23.934132  58.902440


#### 4. Integration: Merge all three sources into a single Master DataFrame. Ensure dates align. Handle missing values resulting from the merge.

In [511]:
zdata = pd.read_sql("SELECT * FROM Zones_Weather", engine)
# zdata_count = pd.read_sql("SELECT COUNT(*) FROM Zones_Weather", engine).iloc[0, 0]
sample = zdata.head()
print(f"\nWEATHER\n{sample}")
print(type(zdata))
# print(zdata_count)

odata = pd.read_sql("SELECT * FROM Ops_Sensor", engine)
# odata_count = pd.read_sql("SELECT COUNT(*) FROM Ops_Sensor", engine).iloc[0, 0]
print(f"\nSENSORS\n{odata.head()}")
# print(odata_count)
query = """
SELECT
    o.timestamp,
    o.Zone,
    o.Pressure_PSI,
    o.Flow_Rate_LPM,
    z.temperature_2m,
    z.relative_humidity_2m,
    z.rain,
    z.weather_code,
    z.precipitation,
    z.apparent_temperature,
    z.surface_pressure

FROM Ops_Sensor AS o

JOIN Zones_Weather AS z
ON o.Zone = z.Zone
AND o.timestamp = z.timestamp;
"""
merged_data = pd.read_sql(query, engine)
print(f"Missing values Count: \n{merged_data.isnull().sum()}")

missing = merged_data[merged_data.isnull().any(axis=1)]  # rows with missing values
print(missing)
merged_data.ffill(inplace=True)  # Master DataFrame
print(f"Missing values Count: \n{merged_data.isnull().sum()}")
# merged_columns = merged_data.columns
# print(f"\nLOOK COLUMNS:\n {merged_columns}, {type(merged_columns)}")
# print(f"\nLOOK COLUMNS WITH LOOP:")
# for column in merged_columns:
#     print(f"- {column}")

# print(f"\nLOOK COLUMN NAME WITH INDEX:\n {merged_columns[1]}", type(merged_columns[1]))
# print(f"\nLOOK COLUMN AS SERIES:\n {merged_data[merged_columns[1]]}", type(merged_data[merged_columns[1]]))
# print(f"\nLOOK COLUMN AS SERIES2:\n {merged_data.Zone}", type(merged_data.Zone))

print(f"\nMERGED (WEATHER+SENSORS)\n{merged_data.head()}")
merged_data.to_csv("merged_ops.csv")


WEATHER
                    timestamp  temperature_2m  relative_humidity_2m  rain  \
0  2026-06-25 00:00:00.000000           15.85             82.637138   0.0   
1  2026-06-25 01:00:00.000000           14.95             86.126381   0.0   
2  2026-06-25 02:00:00.000000           14.60             87.515594   0.0   
3  2026-06-25 03:00:00.000000           14.55             88.086746   0.0   
4  2026-06-25 04:00:00.000000           14.15             88.341743   0.0   

   weather_code  precipitation  apparent_temperature  surface_pressure  \
0           1.0            0.0             15.890408        840.129211   
1           1.0            0.0             14.906767        839.381042   
2           0.0            0.0             14.600861        838.609680   
3           0.0            0.0             14.295141        838.746399   
4           0.0            0.0             13.497416        838.686707   

           Zone  
0  Zone_Central  
1  Zone_Central  
2  Zone_Central  
3  Zone_Cen

#### 5. Analysis: Perform one correlation analysis. (e.g., "Does rainfall correlate with lower throughput?" or "Do holidays increase transaction volume?").

In [512]:
merged_data['timestamp'] = pd.to_datetime(merged_data['timestamp'])
merged_data = merged_data.set_index(['timestamp']).sort_index()
print(merged_data.dtypes, "\n")
temp_vs_throughput  = merged_data['temperature_2m'].corr(merged_data['Flow_Rate_LPM'])
print(f"\nTemperature vs Flow Rate Correlation: {temp_vs_throughput:.2f}")
print()
low_throughput = merged_data.nsmallest(20, 'Flow_Rate_LPM')
print(low_throughput.head())
print(low_throughput.dtypes)
high_throughput = merged_data.nlargest(20, 'Flow_Rate_LPM')


Zone                        str
Pressure_PSI            float64
Flow_Rate_LPM           float64
temperature_2m          float64
relative_humidity_2m    float64
rain                    float64
weather_code            float64
precipitation           float64
apparent_temperature    float64
surface_pressure        float64
dtype: object 


Temperature vs Flow Rate Correlation: 0.01

                             Zone  Pressure_PSI  Flow_Rate_LPM  \
timestamp                                                        
2026-06-25 02:00:00     Zone_East    160.157953     673.194158   
2026-06-26 21:00:00    Zone_South    184.513951     716.486706   
2026-06-30 08:00:00     Zone_East    205.383753     716.968183   
2026-06-27 02:00:00  Zone_Central    201.468924     723.548105   
2026-07-01 11:00:00  Zone_Central    201.464269     743.424002   

                     temperature_2m  relative_humidity_2m  rain  weather_code  \
timestamp                                                                  

In [513]:
import hvplot.pandas


temp_vs_low_throughput = pd.DataFrame(
    {
        'Temperature': low_throughput['temperature_2m'],
        'Throughput': low_throughput['Flow_Rate_LPM']
    }
)

print(temp_vs_low_throughput.index)

temp_vs_low_throughput.hvplot.scatter(x='Temperature', y='Throughput')

DatetimeIndex(['2026-06-25 02:00:00', '2026-06-26 21:00:00',
               '2026-06-30 08:00:00', '2026-06-27 02:00:00',
               '2026-07-01 11:00:00', '2026-06-25 13:00:00',
               '2026-06-27 04:00:00', '2026-06-28 05:00:00',
               '2026-06-25 08:00:00', '2026-06-28 07:00:00',
               '2026-06-25 22:00:00', '2026-07-01 09:00:00',
               '2026-06-25 04:00:00', '2026-06-30 03:00:00',
               '2026-06-27 06:00:00', '2026-06-28 15:00:00',
               '2026-06-28 09:00:00', '2026-06-25 18:00:00',
               '2026-06-29 00:00:00', '2026-06-29 01:00:00'],
              dtype='datetime64[us]', name='timestamp', freq=None)


:Scatter   [Temperature]   (Throughput)

In [514]:
temp_vs_high_throughput = pd.DataFrame(
    {
        'Temperature': high_throughput['temperature_2m'],
        'Throughput': high_throughput['Flow_Rate_LPM']
    }
)

temp_vs_high_throughput.hvplot.scatter(x='Temperature', y='Throughput')

:Scatter   [Temperature]   (Throughput)

#### 6. Documentation: Comment your code heavily, explaining the source of each column.

## CSV FILE
### mock_ops.csv data
- timestamp
- date
- Zone
- Shift
- Pressure_PSI
- Temperature_C
- Flow_Rate_LPM

## API RESPONSE
### Open Meteo API - past 1 month archived data
- timestamp
- temperature_2m
- relative_humidity_2m
- rain
- weather_code
- precipitation
- apparent_temperature
- surface_pressure  

## DATABASE RETRIEVAL
### Merged and aligned data from both mock_ops.csv and API response
#### All of the above fields, selected:
- timestamp
- Zone
- Pressure_PSI
- Flow_Rate_LPM
- temperature_2m
- relative_humidity_2m
- rain
- weather_code
- precipitation
- apparent_temperature
- surface_pressure

My future improvements:
-- arithmetically calculate throughput with appropriate formulas
-- appropriate error handling
-- better plots, transalating the correlation well